Awesome—here’s a full updated version of your script that supports multiple dwells in a single CSV and trains using a PyG DataLoader. I built it to stay close to your original architecture (EdgeConv + node classification), but it now:

Loads all dwells from one CSV,
Splits at the graph (dwell) level into train/val/test,
Standardizes features using only the training graphs,
Builds kNN edges per graph after standardization,
Trains/evaluates via mini-batches of graphs using DataLoader.

You can download the file here:

Script: gnn_edgeconv_multi_dwells.py

What changed (and why)

- Graph-level splits: Instead of node masks within a single dwell, we split the list of dwell graphs into train/val/test sets. This aligns naturally with how PyG’s DataLoader batches multiple graphs (it internally creates a big disconnected graph and keeps track of graph membership).  
- Train-only standardization: Feature mean/std are computed over all nodes from training graphs only, then applied to val/test—this avoids data leakage and mirrors your single-dwell train-only normalization.  
- kNN edges per graph: k is applied per graph after standardization. For very small dwells, k_eff = min(k, N-1) avoids runtime errors.  
- Loss & metrics over batches: We compute BCE loss, accuracy, precision/recall/F1 over all nodes in the batch, which is appropriate for node-level classification.  
- Class imbalance: pos_weight is computed across the training graphs to balance background vs group nodes (same idea you used, now extended to multi-graph).

CSV schema (expected)  
Your single CSV should include:  

- dwell_id – identifying each dwell (string or int)  
- label – 0 for background, 1 for group (node-level)  
- Features (default in the script):  
x_sensor, y_sensor, z_sensor, x_target, y_target, LOS_Velocity  

If your column names differ, adjust feature_cols in main() and --knn-on accordingly.

Great update adding z_target—moving to true 3D geometry will help the network learn the collection geometry and motion features more faithfully.
Below are three edge-construction options and my recommendation for your case, plus the exact code changes to make it work cleanly with --knn-on.

Recommendation

Default --knn-on: use relative 3D coordinates (x_rel, y_rel, z_rel) from sensor to target, not raw target or sensor positions.
This enforces edges that reflect the line-of-sight (LoS) geometry while remaining invariant to global translations.
Always build edges in 3D (kNN on [x_rel, y_rel, z_rel]) and consider adding a star (sensor→detections) edge set if you want the network to explicitly see the sensor-to-node topology.


Why relative coordinates?
If edges are built on raw (x_target, y_target, z_target), the kNN neighborhood depends on the absolute coordinate frame and sensor position. Using relative vectors:  

x_rel = x_target - x_sensor
y_rel = y_target - y_sensor
z_rel = z_target - z_sensor

means your neighborhoods reflect true 3D proximity in the sensor’s local frame, which is what defines the LoS rays and collection geometry. This also reduces sensitivity to global translations of the platform/scene.  

Edge construction options
Option A — 3D kNN on relative positions (Recommended baseline)

Build kNN using [x_rel, y_rel, z_rel].
Keeps your current EdgeConv pipeline intact.
Set default --knn-on to ["x_rel", "y_rel", "z_rel"].

Option B — Hybrid edges: Star + Local kNN

Star edges: connect a sensor node to every detection (one hop LoS).
Local kNN: among detections using [x_rel, y_rel, z_rel] to capture convoy/group spatial cohesion.
Concatenate both into edge_index.
This gives the network an explicit notion of the sensor ray and local target interactions.

Option C — Radius graph in 3D

Use radius_graph(x_rel, r=R) instead of kNN.
More physical when you know an interaction range (e.g., group spacing).


Code changes
1) Compute relative features in the loader
Add x_rel, y_rel, z_rel to each graph’s Data.x and to the allowed feature list:


In [ ]:

# In load_multi_dwell_graphs(...)
x_np = gdf[feature_cols].values.astype(np.float32)
# Build relative columns on the fly
x_sensor = gdf[["x_sensor", "y_sensor", "z_sensor"]].values.astype(np.float32)
x_target = gdf[["x_target", "y_target", "z_target"]].values.astype(np.float32)
x_rel = (x_target - x_sensor)  # shape [N, 3]

# Concatenate: [original features + x_rel]
# New order example: [sensor(3) + target(3) + LOS + rel(3)]
x_np = np.concatenate([x_np, x_rel], axis=1)

data = Data(
    x=torch.from_numpy(x_np),
    y=torch.from_numpy(y_np),
)
data.dwell_id = dwell_id


Then update your feature_cols to include the rel-components (for clarity when saving metadata):

In [ ]:

feature_cols = [
    "x_sensor", "y_sensor", "z_sensor",
    "x_target", "y_target", "z_target",
    "LOS_Velocity",
    "x_rel", "y_rel", "z_rel"  # new
]


If you prefer not to expand feature_cols, you can still compute x_rel inside build_edges_for_graph using the columns you have; I include it in Data.x because it’s also a useful node feature for EdgeConv.



2) Make default --knn-on the relative 3D columns
Change the parser default:

In [ ]:

parser.add_argument("--knn-on", type=str, nargs="+",
    default=["x_rel", "y_rel", "z_rel"],
    help="Columns used to build kNN graph (default: x_rel y_rel z_rel)."
)


And in build_edges_for_graph(...) you already map names to indices:

In [ ]:

knn_idx = [feature_cols.index(c) for c in knn_on_cols]
x_for_knn = g.x[:, knn_idx]
edge_index = knn_graph(x_for_knn, k=k_eff, loop=False)  # 3D neighborhoods


3) (Optional) Add a sensor node and star edges
If you want explicit sensor→node edges, create a synthetic sensor node per dwell and connect it to all detections:

In [ ]:

def add_sensor_star_edges(g: Data, feature_cols: List[str]):
    # Build a synthetic sensor node at index 0 (or at the end)
    # Sensor feature vector can be zeros or mean; we mostly need it as a hub.
    N = g.x.size(0)
    sensor_feat = torch.zeros((1, g.x.size(1)), dtype=g.x.dtype)
    g.x = torch.cat([sensor_feat, g.x], dim=0)  # prepend
    # Star edges from sensor node (index 0) to all detection nodes [1..N]
    src = torch.zeros(N, dtype=torch.long)
    dst = torch.arange(1, N+1, dtype=torch.long)
    star_edges = torch.vstack([torch.cat([src, dst]), torch.cat([dst, src])])  # bidirectional
    # Merge with existing kNN edges (remember kNN edges built on original indices; shift them by +1)
    if g.edge_index is not None and g.edge_index.numel() > 0:
        g.edge_index = g.edge_index + 1  # shift all node ids by +1
        g.edge_index = torch.cat([g.edge_index, star_edges], dim=1)
    else:
        g.edge_index = star_edges


Here’s your updated multi-dwell PyTorch Geometric script with:
✅ Automatic computation of relative features (x_rel, y_rel, z_rel)
✅ Addition of a sensor node per graph
✅ Star edges connecting the sensor node to all detections
✅ Combined kNN edges (based on relative 3D positions)
✅ Default --knn-on set to ["x_rel", "y_rel", "z_rel"]
✅ Updated feature_cols to include relative features

Key Changes

Loader now computes relative features and appends them to node features.
Edge building:

kNN uses relative coordinates.
Adds a synthetic sensor node and star edges for explicit sensor connectivity.


Feature columns:
["x_sensor", "y_sensor", "z_sensor",
 "x_target", "y_target", "z_target",
 "LOS_Velocity", "x_rel", "y_rel", "z_rel"]


Default --knn-on: ["x_rel", "y_rel", "z_rel"].


Run Example

In [ ]:
#Enter into a PS terminal on VSCode:

python gnn_inference_multi_dwells_metrics.py `
  --csv "C:/Users/charlie.burgwardt/OneDrive - NV5/GMTI/Data/new_dwells.csv" `
  --checkpoint "edgeconv_multi_dwells_sensor_fixed_best.pt" `
  --device cpu `
  --out "predictions.csv" `
  --metrics "per_dwell_metrics.csv"
